# Chapter 2: Using Transformers

In [1]:
# Setup: install deps and authenticate to the Hub. Re-run on every fresh runtime.
%pip install -q transformers datasets evaluate accelerate  # add peft when needed

import os
from getpass import getpass
from huggingface_hub import whoami
try:
    from google.colab import userdata  # works only in the Colab web UI
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN_COURSE")
except Exception as e:
    print("Colab secret not read:", repr(e))
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Paste your HF token (hidden, not saved): ")
HF_USER = whoami()["name"]  # HF libraries read HF_TOKEN from the env; this checks it works
print("Logged in to the Hub as", HF_USER)

Colab secret not read: TimeoutException('Requesting secret HF_TOKEN_COURSE timed out. Secrets can only be fetched when running from the Colab UI.')
Logged in to the Hub as pedropalvareez


## 2.2 Behind the pipeline

Course page: https://huggingface.co/learn/llm-course/chapter2/2

The Ch1 `pipeline` call hides three steps. This section runs them by hand:

1. **Tokenizer**: text → token IDs (integers).
2. **Model**: token IDs → logits (one raw score per class).
3. **Postprocessing**: logits → probabilities (softmax) → label.

In [2]:
# The pipeline we will take apart.
# v5 adaptation: model named explicitly (project rule); it is the same checkpoint the course gets by default.
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

### Preprocessing with a tokenizer

In [4]:
# AutoTokenizer reads the checkpoint's config and loads the matching tokenizer class.
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [5]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
# padding: make both rows the same length (tensors are rectangular)
# truncation: cut anything over the model's max length (512)
# return_tensors="pt": PyTorch tensors instead of Python lists
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


### Going through the model

In [3]:
# AutoModel = the transformer body only, no task head.
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### A high-dimensional vector?

In [6]:
# **inputs unpacks the dict into keyword arguments: model(input_ids=..., attention_mask=...)
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)  # (batch size, sequence length, hidden size)

torch.Size([2, 16, 768])


### Model heads: Making sense out of numbers

In [7]:
# Same body + a classification head: a linear layer mapping the [CLS] vector (768) to 2 scores.
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [8]:
print(outputs.logits.shape)  # (batch size, number of labels)

torch.Size([2, 2])


### Postprocessing the output

In [9]:
print(outputs.logits)  # raw scores (logits): any real number, not probabilities

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


In [10]:
# softmax turns each row of logits into probabilities: exp(z_i) / sum_j exp(z_j)
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)  # dim=-1: across the labels, per sentence
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


In [11]:
model.config.id2label  # which column is which label

{0: 'NEGATIVE', 1: 'POSITIVE'}

## 2.3 Models

Course page: https://huggingface.co/learn/llm-course/chapter2/3

How to load a model from the Hub, save it, upload it, and what the tokenizer must hand the model (rectangular tensors of IDs).

### Creating a Transformer

In [12]:
# AutoModel reads the checkpoint's config.json and picks the right architecture class.
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# Same thing, naming the class directly. Only works if you know the checkpoint is a BERT.
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-cased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Loading and saving

In [14]:
# Writes config.json (architecture) + model.safetensors (weights) to a folder on the Colab disk.
model.save_pretrained("directory_on_my_computer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
!ls directory_on_my_computer

config.json  model.safetensors


In [16]:
# from_pretrained accepts a local folder as well as a Hub name.
from transformers import AutoModel

model = AutoModel.from_pretrained("directory_on_my_computer")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Course step skipped: `notebook_login()` / `huggingface-cli login`. The setup cell already logged us in through `HF_TOKEN`.

In [17]:
# Upload to the Hub. Project rule: private repos.
model.push_to_hub("my-awesome-model", private=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...7btp115/model.safetensors:   7%|7         | 31.9MB /  433MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/pedropalvareez/my-awesome-model/commit/98e245685fdfe9365632fab01f7b52371b9179d8', commit_message='Upload model', commit_description='', oid='98e245685fdfe9365632fab01f7b52371b9179d8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pedropalvareez/my-awesome-model', endpoint='https://huggingface.co', repo_type='model', repo_id='pedropalvareez/my-awesome-model'), pr_revision=None, pr_num=None)

In [19]:
# Load it back by "username/repo". HF_USER comes from the setup cell.
from transformers import AutoModel

model = AutoModel.from_pretrained(f"{HF_USER}/my-awesome-model")

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Encoding text

In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

encoded_input = tokenizer("Hello, I'm a single sentence!")
print(encoded_input)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

{'input_ids': [101, 8667, 117, 146, 112, 182, 170, 1423, 5650, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [21]:
# IDs back to text. [CLS] and [SEP] were added by the tokenizer: BERT was pretrained with them.
tokenizer.decode(encoded_input["input_ids"])

"[CLS] Hello, I ' m a single sentence! [SEP]"

In [22]:
# Two strings as two arguments = ONE sentence pair: [CLS] A [SEP] B [SEP].
# The course page's printed output (two separate lists) is wrong. Expect one list,
# with token_type_ids 0 for sentence A and 1 for sentence B.
encoded_input = tokenizer("How are you?", "I'm fine, thank you!")
print(encoded_input)

{'input_ids': [101, 1731, 1132, 1128, 136, 102, 146, 112, 182, 2503, 117, 6243, 1128, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [23]:
encoded_input = tokenizer("How are you?", "I'm fine, thank you!", return_tensors="pt")
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  136,  102,  146,  112,  182, 2503,  117, 6243,
         1128,  106,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


### Padding inputs

In [24]:
# A LIST of two strings = two separate sentences. padding=True pads the shorter one with 0 ([PAD]).
encoded_input = tokenizer(
    ["How are you?", "I'm fine, thank you!"], padding=True, return_tensors="pt"
)
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  136,  102,    0,    0,    0,    0],
        [ 101,  146,  112,  182, 2503,  117, 6243, 1128,  106,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


### Truncating inputs

In [25]:
encoded_input = tokenizer(
    "This is a very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very long sentence.",
    truncation=True,
)
print(encoded_input["input_ids"])  # under 512 tokens, so nothing is cut here

[101, 1188, 1110, 170, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1263, 5650, 119, 102]


In [26]:
# max_length=5 forces the cut: [CLS] + 3 tokens + [SEP].
encoded_input = tokenizer(
    ["How are you?", "I'm fine, thank you!"],
    padding=True,
    truncation=True,
    max_length=5,
    return_tensors="pt",
)
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  102],
        [ 101,  146,  112,  182,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1]])}


### Adding special tokens

In [27]:
encoded_input = tokenizer("How are you?")
print(encoded_input["input_ids"])
tokenizer.decode(encoded_input["input_ids"])

[101, 1731, 1132, 1128, 136, 102]


'[CLS] How are you? [SEP]'

### Why is all of this necessary?

In [28]:
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]

In [29]:
encoded_sequences = [
    [101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102],
    [101, 1045, 5223, 2023, 2061, 2172, 999, 102],
]

In [31]:
# EXPECTED ERROR: the rows have 16 and 8 IDs, and a tensor must be rectangular.
# The course page skips this; it is exactly the problem padding solves.
import torch

model_inputs = torch.tensor(encoded_sequences)

ValueError: expected sequence of length 16 at dim 1 (got 8)

In [32]:
# Fix: pad the short row with 0 ([PAD]) up to 16, then convert.
encoded_sequences[1] += [0] * (len(encoded_sequences[0]) - len(encoded_sequences[1]))
model_inputs = torch.tensor(encoded_sequences)
print(model_inputs.shape)

torch.Size([2, 16])


### Using the tensors as inputs to the model

In [33]:
output = model(model_inputs)
print(output.last_hidden_state.shape)  # (2 sentences, 16 tokens, 768)

torch.Size([2, 16, 768])


## 2.4 Tokenizers

Course page: https://huggingface.co/learn/llm-course/chapter2/4

Models read numbers, not text. A tokenizer splits text into pieces and maps each piece to an integer ID from a fixed vocabulary. The design question is what a "piece" is.

### Word-based

In [34]:
# Split on whitespace. Simple, but every word form ("dog", "dogs", "dog's") is a separate vocabulary entry,
# the vocabulary explodes, and any unseen word becomes [UNK].
tokenized_text = "Jim Henson was a puppeteer".split()
print(tokenized_text)

['Jim', 'Henson', 'was', 'a', 'puppeteer']


### Character-based

Split into single characters. Tiny vocabulary and almost no unknown tokens, but each character carries little meaning and sequences get very long (a model's context fills up fast).

### Subword tokenization

The middle ground used by modern models: frequent words stay whole, rare words split into meaningful pieces (`tokenization` → `token` + `##ization`). Small vocabulary, few unknowns, short sequences.

### And more!

Main algorithms: **Byte-level BPE** (GPT-2), **WordPiece** (BERT), **SentencePiece / Unigram** (multilingual models). Chapter 6 of the course covers them.

### Loading and saving

In [35]:
# Architecture-specific class, same pattern as BertModel.
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

In [36]:
# AutoTokenizer picks the class from the checkpoint. Use this one by default.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [37]:
tokenizer("Using a Transformer network is simple")

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [38]:
# Writes the vocabulary + tokenizer settings (tokenizer.json, tokenizer_config.json, ...).
tokenizer.save_pretrained("directory_on_my_computer")
!ls directory_on_my_computer

config.json  model.safetensors	tokenizer_config.json  tokenizer.json


### Encoding

Text → IDs happens in two steps: **tokenize** (text → tokens), then **convert** (tokens → IDs via the vocabulary). `tokenizer(...)` does both at once and adds the special tokens; below we do them by hand.

### Tokenization

In [39]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

print(tokens)  # "##" = this piece continues the previous token

['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']


### From tokens to input IDs

In [40]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)  # same as tokenizer(sequence)["input_ids"] minus [CLS]=101 and [SEP]=102

[7993, 170, 13809, 23763, 2443, 1110, 3014]


### Decoding

In [41]:
# IDs → text. Merges "##" pieces back into words. This is how generated IDs become readable text.
decoded_string = tokenizer.decode([7993, 170, 11303, 1200, 2443, 1110, 3014])
print(decoded_string)

Using a transformer network is simple


## 2.5 Handling multiple sequences

Course page: https://huggingface.co/learn/llm-course/chapter2/5

Real inputs come in batches of sentences with different lengths. Three tools make that work: a **batch dimension**, **padding**, and the **attention mask**. Ch3 training is batches all the way down.

### Models expect a batch of inputs

In [42]:
# EXPECTED ERROR: input_ids is 1-D (just the IDs). The model wants 2-D: (batch size, sequence length).
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)
# This line will fail.
model(input_ids)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (14) must match the size of tensor b (512) at non-singleton dimension 1

In [43]:
# The tokenizer adds the batch dimension for you: note the double brackets [[...]].
tokenized_inputs = tokenizer(sequence, return_tensors="pt")
print(tokenized_inputs["input_ids"])

tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]])


In [44]:
# Fix: wrap ids in a list -> shape (1, 14), a batch of one.
# Logits differ from section 2.2 ([-1.5607, 1.6123]): tokenize() + convert_tokens_to_ids() skip [CLS] and [SEP].
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])
print("Input IDs:", input_ids)

output = model(input_ids)
print("Logits:", output.logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [45]:
batched_ids = [ids, ids]  # a batch of two identical sequences

**✏️ Try it out:** convert `batched_ids` into a tensor and pass it through the model. You should get the same logits as above, twice.

In [46]:
# Your turn: two lines. Hint: same pattern as the cell above, with batched_ids instead of [ids].


### Padding the inputs

In [47]:
# Can't become a tensor: rows of length 3 and 2.
batched_ids = [
    [200, 200, 200],
    [200, 200]
]

In [48]:
# Pad the short row with a padding token so the batch is rectangular.
padding_id = 100

batched_ids = [
    [200, 200, 200],
    [200, 200, padding_id],
]

In [49]:
# Compare each sentence alone vs. both in a padded batch.
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],  # the model's real padding ID (0 for this tokenizer)
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)  # row 2 will NOT match line 2: attention also reads the [PAD]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


### Attention masks

Attention computes each token's new vector as a weighted average over all tokens, with weights = softmax(scores). The mask sets the score of masked tokens to −∞ before the softmax, so their weight becomes exactly 0 and [PAD] contributes nothing.

In [50]:
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

attention_mask = [
    [1, 1, 1],
    [1, 1, 0],  # 0 = ignore this position
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)  # row 2 now matches sequence2 alone

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


**✏️ Try it out:** tokenize the two section 2.2 sentences by hand (`tokenize` + `convert_tokens_to_ids`) and check their logits against section 2.2. Then batch them with padding, build the attention mask, and check you get the same results.

Hint: section 2.2 got `[-1.5607, 1.6123]` and `[4.1692, -3.3464]`. To match, your hand-made IDs need what the tokenizer adds on its own: `tokenizer.cls_token_id` at the start and `tokenizer.sep_token_id` at the end.

In [ ]:
# Your turn.


### Longer sequences

Models have a maximum length (512 tokens for BERT/DistilBERT). Longer input either needs a long-context model (Longformer, LED) or truncation. The course shows:

```python
sequence = sequence[:max_sequence_length]
```

That slices characters of a string, and `max_sequence_length` is never defined, so it is illustrative only. In practice use `tokenizer(..., truncation=True, max_length=...)` as in section 2.3, which cuts in tokens.

In [51]:
tokenizer.model_max_length  # the limit truncation=True cuts to by default

512

## 2.6 Putting it all together

Course page: https://huggingface.co/learn/llm-course/chapter2/6

In the last few sections, we've been trying our best to do most of the work by hand. We've explored how tokenizers work and looked at tokenization, conversion to input IDs, padding, truncation, and attention masks.

However, as we saw in section 2, the 🤗 Transformers API can handle all of this for us with a high-level function that we'll dive into here. When you call your `tokenizer` directly on the sentence, you get back inputs that are ready to pass through your model:

In [57]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

model_inputs = tokenizer(sequence)

In [58]:
# 👀 Look (not in the course): what the tokenizer returned for one sentence.
model_inputs

{'input_ids': [101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Here, the `model_inputs` variable contains everything that's necessary for a model to operate well. For DistilBERT, that includes the input IDs as well as the attention mask. Other models that accept additional inputs will also have those output by the `tokenizer` object.

As we'll see in some examples below, this method is very powerful. First, it can tokenize a single sequence:

In [59]:
sequence = "I've been waiting for a HuggingFace course my whole life."

model_inputs = tokenizer(sequence)

It also handles multiple sequences at a time, with no change in the API:

In [60]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

model_inputs = tokenizer(sequences)

In [61]:
# 👀 Look (not in the course): two sentences, two lists of different lengths (16 and 6 IDs). Not a tensor yet.
model_inputs["input_ids"]

[[101,
  1045,
  1005,
  2310,
  2042,
  3403,
  2005,
  1037,
  17662,
  12172,
  2607,
  2026,
  2878,
  2166,
  1012,
  102],
 [101, 2061, 2031, 1045, 999, 102]]

It can pad according to several objectives:

In [62]:
# Will pad the sequences up to the maximum sequence length
model_inputs = tokenizer(sequences, padding="longest")

# Will pad the sequences up to the model max length
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(sequences, padding="max_length")

# Will pad the sequences up to the specified max length
model_inputs = tokenizer(sequences, padding="max_length", max_length=8)

In [63]:
# 👀 Look (not in the course): tokens per sentence under each padding option.
# The cell above overwrote model_inputs three times, so we re-run each option here.
print("longest:              ", [len(x) for x in tokenizer(sequences, padding="longest")["input_ids"]])
print("max_length:           ", [len(x) for x in tokenizer(sequences, padding="max_length")["input_ids"]])
print("max_length, max 8:    ", [len(x) for x in tokenizer(sequences, padding="max_length", max_length=8)["input_ids"]])
# Last line: padding never cuts, so the 16-token sentence stays 16 -> still not rectangular.

longest:               [16, 16]
max_length:            [512, 512]
max_length, max 8:     [16, 8]


It can also truncate sequences:

In [64]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

# Will truncate the sequences that are longer than the model max length
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(sequences, truncation=True)

# Will truncate the sequences that are longer than the specified max length
model_inputs = tokenizer(sequences, max_length=8, truncation=True)

In [65]:
# 👀 Look (not in the course): tokens per sentence under each truncation option.
print("truncation (512):     ", [len(x) for x in tokenizer(sequences, truncation=True)["input_ids"]])
print("truncation, max 8:    ", [len(x) for x in tokenizer(sequences, max_length=8, truncation=True)["input_ids"]])
print(tokenizer.decode(tokenizer(sequences, max_length=8, truncation=True)["input_ids"][0]))  # what survives the cut: [SEP] is kept

truncation (512):      [16, 6]
truncation, max 8:     [8, 6]
[CLS] i ' ve been waiting for [SEP]


The `tokenizer` object can handle the conversion to specific framework tensors, which can then be directly sent to the model. For example, in the following code sample we are prompting the tokenizer to return tensors from the different frameworks — `"pt"` returns PyTorch tensors and `"np"` returns NumPy arrays:

In [66]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

# Returns PyTorch tensors
model_inputs = tokenizer(sequences, padding=True, return_tensors="pt")

# Returns NumPy arrays
model_inputs = tokenizer(sequences, padding=True, return_tensors="np")

In [67]:
# 👀 Look (not in the course): same data, two container types.
print(type(tokenizer(sequences, padding=True, return_tensors="pt")["input_ids"]))
print(type(tokenizer(sequences, padding=True, return_tensors="np")["input_ids"]))
model_inputs["input_ids"].shape  # (2 sentences, 16 tokens): rectangular thanks to padding

<class 'torch.Tensor'>
<class 'numpy.ndarray'>


(2, 16)

### Special tokens

If we take a look at the input IDs returned by the tokenizer, we will see they are a tiny bit different from what we had earlier:

In [68]:
sequence = "I've been waiting for a HuggingFace course my whole life."

model_inputs = tokenizer(sequence)
print(model_inputs["input_ids"])

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102]
[1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]


One token ID was added at the beginning, and one at the end. Let's decode the two sequences of IDs above to see what this is about:

In [69]:
print(tokenizer.decode(model_inputs["input_ids"]))
print(tokenizer.decode(ids))

[CLS] i ' ve been waiting for a huggingface course my whole life. [SEP]
i ' ve been waiting for a huggingface course my whole life.


The tokenizer added the special word `[CLS]` at the beginning and the special word `[SEP]` at the end. This is because the model was pretrained with those, so to get the same results for inference we need to add them as well. Note that some models don't add special words, or add different ones; models may also add these special words only at the beginning, or only at the end. In any case, the tokenizer knows which ones are expected and will deal with this for you.

### Wrapping up: From tokenizer to model

Now that we've seen all the individual steps the `tokenizer` object uses when applied on texts, let's see one final time how it can handle multiple sequences (padding!), very long sequences (truncation!), and multiple types of tensors with its main API:

In [70]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

tokens = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")
output = model(**tokens)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [71]:
# 👀 Look (not in the course): close the loop from 2.2: logits -> softmax -> label.
probs = torch.nn.functional.softmax(output.logits, dim=-1)
print(output.logits)
for s, p in zip(sequences, probs):
    print(model.config.id2label[p.argmax().item()], round(p.max().item(), 4), "|", s)

tensor([[-1.5607,  1.6123],
        [-3.6183,  3.9137]], grad_fn=<AddmmBackward0>)
POSITIVE 0.9598 | I've been waiting for a HuggingFace course my whole life.
POSITIVE 0.9995 | So have I!


## 2.8 Optimized Inference Deployment

In this section, we'll explore advanced frameworks for optimizing LLM deployments: Text Generation Inference (TGI), vLLM, and llama.cpp. These applications are primarily used in production environments to serve LLMs to users. This section focuses on how to deploy these frameworks in production rather than how to use them for inference on a single machine.

We'll cover how these tools maximize inference efficiency and simplify production deployments of Large Language Models.

> **Note (not in the course):** this is a reading section. The page's code needs Docker or a running server, and Colab has neither, so its code blocks stay as markdown. Out of date or wrong on the page: llama.cpp now builds with CMake and the server binary is `llama-server` (not `make` / `./server`); `client.generate(...)` is not an `InferenceClient` method (use `text_generation`); `LLM(engine_args=...)` is not valid vLLM (pass the arguments to `LLM(...)` directly).
>
> The runnable 👀 cells use plain `transformers` on CPU with the page's model (SmolLM2-360M-Instruct) to show the same ideas: sampling, repetition, stopping, KV cache, quantization.

### Framework Selection Guide

TGI, vLLM, and llama.cpp serve similar purposes but have distinct characteristics that make them better suited for different use cases. Let's look at the key differences between them, focusing on performance and integration.

#### Memory Management and Performance

**TGI** is designed to be stable and predictable in production, using fixed sequence lengths to keep memory usage consistent. TGI manages memory using Flash Attention 2 and continuous batching techniques. This means it can process attention calculations very efficiently and keep the GPU busy by constantly feeding it work. The system can move parts of the model between CPU and GPU when needed, which helps handle larger models. 

![Flash Attention](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/tgi/flash-attn.png)

> 💡 **How Flash Attention Works**

Flash Attention is a technique that optimizes the attention mechanism in transformer models by addressing memory bandwidth bottlenecks. As discussed earlier in [Chapter 1.8](https://huggingface.co/learn/llm-course/chapter1/8), the attention mechanism has quadratic complexity and memory usage, making it inefficient for long sequences.

The key innovation is in how it manages memory transfers between High Bandwidth Memory (HBM) and faster SRAM cache. Traditional attention repeatedly transfers data between HBM and SRAM, creating bottlenecks by leaving the GPU idle. Flash Attention loads data once into SRAM and performs all calculations there, minimizing expensive memory transfers. 

While the benefits are most significant during training, Flash Attention's reduced VRAM usage and improved efficiency make it valuable for inference as well, enabling faster and more scalable LLM serving.

**vLLM** takes a different approach by using PagedAttention. Just like how a computer manages its memory in pages, vLLM splits the model's memory into smaller blocks. This clever system means it can handle different-sized requests more flexibly and doesn't waste memory space. It's particularly good at sharing memory between different requests and reduces memory fragmentation, which makes the whole system more efficient.

> 💡 **How PagedAttention Works**

PagedAttention is a technique that addresses another critical bottleneck in LLM inference: KV cache memory management. As discussed in [Chapter 1.8](https://huggingface.co/learn/llm-course/chapter1/8), during text generation, the model stores attention keys and values (KV cache) for each generated token to reduce redundant computations. The KV cache can become enormous, especially with long sequences or multiple concurrent requests.

vLLM's key innovation lies in how it manages this cache:

1. **Memory Paging**: Instead of treating the KV cache as one large block, it's divided into fixed-size "pages" (similar to virtual memory in operating systems).
2. **Non-contiguous Storage**: Pages don't need to be stored contiguously in GPU memory, allowing for more flexible memory allocation.
3. **Page Table Management**: A page table tracks which pages belong to which sequence, enabling efficient lookup and access.
4. **Memory Sharing**: For operations like parallel sampling, pages storing the KV cache for the prompt can be shared across multiple sequences.

The PagedAttention approach can lead to up to 24x higher throughput compared to traditional methods, making it a game-changer for production LLM deployments. If you want to go really deep into how PagedAttention works, you can read the [the guide from the vLLM documentation](https://docs.vllm.ai/en/latest/design/kernel/paged_attention.html).

**llama.cpp** is a highly optimized C/C++ implementation originally designed for running LLaMA models on consumer hardware. It focuses on CPU efficiency with optional GPU acceleration and is ideal for resource-constrained environments. llama.cpp uses quantization techniques to reduce model size and memory requirements while maintaining good performance. It implements optimized kernels for various CPU architectures and supports basic KV cache management for efficient token generation.

> 💡 **How llama.cpp Quantization Works**

Quantization in llama.cpp reduces the precision of model weights from 32-bit or 16-bit floating point to lower precision formats like 8-bit integers (INT8), 4-bit, or even lower. This significantly reduces memory usage and improves inference speed with minimal quality loss.

Key quantization features in llama.cpp include:
1. **Multiple Quantization Levels**: Supports 8-bit, 4-bit, 3-bit, and even 2-bit quantization
2. **GGML/GGUF Format**: Uses custom tensor formats optimized for quantized inference
3. **Mixed Precision**: Can apply different quantization levels to different parts of the model
4. **Hardware-Specific Optimizations**: Includes optimized code paths for various CPU architectures (AVX2, AVX-512, NEON)

This approach enables running billion-parameter models on consumer hardware with limited memory, making it perfect for local deployments and edge devices.

#### Deployment and Integration

Let's move on to the deployment and integration differences between the frameworks.

**TGI** excels in enterprise-level deployment with its production-ready features. It comes with built-in Kubernetes support and includes everything you need for running in production, like monitoring through Prometheus and Grafana, automatic scaling, and comprehensive safety features. The system also includes enterprise-grade logging and various protective measures like content filtering and rate limiting to keep your deployment secure and stable.

**vLLM** takes a more flexible, developer-friendly approach to deployment. It's built with Python at its core and can easily replace OpenAI's API in your existing applications. The framework focuses on delivering raw performance and can be customized to fit your specific needs. It works particularly well with Ray for managing clusters, making it a great choice when you need high performance and adaptability.

**llama.cpp** prioritizes simplicity and portability. Its server implementation is lightweight and can run on a wide range of hardware, from powerful servers to consumer laptops and even some high-end mobile devices. With minimal dependencies and a simple C/C++ core, it's easy to deploy in environments where installing Python frameworks would be challenging. The server provides an OpenAI-compatible API while maintaining a much smaller resource footprint than other solutions.

### Getting Started

Let's explore how to use these frameworks for deploying LLMs, starting with installation and basic setup.

#### Installation and Basic Setup

**TGI**

TGI is easy to install and use, with deep integration into the Hugging Face ecosystem.

First, launch the TGI server using Docker:

```sh
docker run --gpus all \
    --shm-size 1g \
    -p 8080:80 \
    -v ~/.cache/huggingface:/data \
    ghcr.io/huggingface/text-generation-inference:latest \
    --model-id HuggingFaceTB/SmolLM2-360M-Instruct
```

Then interact with it using Hugging Face's InferenceClient:

```python
from huggingface_hub import InferenceClient

# Initialize client pointing to TGI endpoint
client = InferenceClient(
    model="http://localhost:8080",  # URL to the TGI server
)

# Text generation
response = client.text_generation(
    "Tell me a story",
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.95,
    details=True,
    stop_sequences=[],
)
print(response.generated_text)

# For chat format
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)
```

Alternatively, you can use the OpenAI client:

```python
from openai import OpenAI

# Initialize client pointing to TGI endpoint
client = OpenAI(
    base_url="http://localhost:8080/v1",  # Make sure to include /v1
    api_key="not-needed",  # TGI doesn't require an API key by default
)

# Chat completion
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)
```

**llama.cpp**

llama.cpp is easy to install and use, requiring minimal dependencies and supporting both CPU and GPU inference.

First, install and build llama.cpp:

```sh
# Clone the repository
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp

# Build the project
make

# Download the SmolLM2-1.7B-Instruct-GGUF model
curl -L -O https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct-GGUF/resolve/main/smollm2-1.7b-instruct.Q4_K_M.gguf
```

Then, launch the server (with OpenAI API compatibility):

```sh
# Start the server
./server \
    -m smollm2-1.7b-instruct.Q4_K_M.gguf \
    --host 0.0.0.0 \
    --port 8080 \
    -c 4096 \
    --n-gpu-layers 0  # Set to a higher number to use GPU
```

Interact with the server using Hugging Face's InferenceClient:

```python
from huggingface_hub import InferenceClient

# Initialize client pointing to llama.cpp server
client = InferenceClient(
    model="http://localhost:8080/v1",  # URL to the llama.cpp server
    token="sk-no-key-required",  # llama.cpp server requires this placeholder
)

# Text generation
response = client.text_generation(
    "Tell me a story",
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.95,
    details=True,
)
print(response.generated_text)

# For chat format
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)
```

Alternatively, you can use the OpenAI client:

```python
from openai import OpenAI

# Initialize client pointing to llama.cpp server
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="sk-no-key-required",  # llama.cpp server requires this placeholder
)

# Chat completion
response = client.chat.completions.create(
    model="smollm2-1.7b-instruct",  # Model identifier can be anything as server only loads one model
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)
```

**vLLM**

vLLM is easy to install and use, with both OpenAI API compatibility and a native Python interface.

First, launch the vLLM OpenAI-compatible server:

```sh
python -m vllm.entrypoints.openai.api_server \
    --model HuggingFaceTB/SmolLM2-360M-Instruct \
    --host 0.0.0.0 \
    --port 8000
```

Then interact with it using Hugging Face's InferenceClient:

```python
from huggingface_hub import InferenceClient

# Initialize client pointing to vLLM endpoint
client = InferenceClient(
    model="http://localhost:8000/v1",  # URL to the vLLM server
)

# Text generation
response = client.text_generation(
    "Tell me a story",
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.95,
    details=True,
)
print(response.generated_text)

# For chat format
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)
```

Alternatively, you can use the OpenAI client:

```python
from openai import OpenAI

# Initialize client pointing to vLLM endpoint
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",  # vLLM doesn't require an API key by default
)

# Chat completion
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)
```

#### Basic Text Generation

Let's look at examples of text generation with the frameworks:

**TGI**

First, deploy TGI with advanced parameters:
```sh
docker run --gpus all \
    --shm-size 1g \
    -p 8080:80 \
    -v ~/.cache/huggingface:/data \
    ghcr.io/huggingface/text-generation-inference:latest \
    --model-id HuggingFaceTB/SmolLM2-360M-Instruct \
    --max-total-tokens 4096 \
    --max-input-length 3072 \
    --max-batch-total-tokens 8192 \
    --waiting-served-ratio 1.2
```

Use the InferenceClient for flexible text generation:

```python
from huggingface_hub import InferenceClient

client = InferenceClient(model="http://localhost:8080")

# Advanced parameters example
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a creative storyteller."},
        {"role": "user", "content": "Write a creative story"},
    ],
    temperature=0.8,
    max_tokens=200,
    top_p=0.95,
)
print(response.choices[0].message.content)

# Raw text generation
response = client.text_generation(
    "Write a creative story about space exploration",
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.95,
    repetition_penalty=1.1,
    do_sample=True,
    details=True,
)
print(response.generated_text)
```

Or use the OpenAI client:
```python
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")

# Advanced parameters example
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=[
        {"role": "system", "content": "You are a creative storyteller."},
        {"role": "user", "content": "Write a creative story"},
    ],
    temperature=0.8,  # Higher for more creativity
)
print(response.choices[0].message.content)
```

**llama.cpp**

For llama.cpp, you can set advanced parameters when launching the server:

```sh
./server \
    -m smollm2-1.7b-instruct.Q4_K_M.gguf \
    --host 0.0.0.0 \
    --port 8080 \
    -c 4096 \            # Context size
    --threads 8 \        # CPU threads to use
    --batch-size 512 \   # Batch size for prompt evaluation
    --n-gpu-layers 0     # GPU layers (0 = CPU only)
```

Use the InferenceClient:

```python
from huggingface_hub import InferenceClient

client = InferenceClient(model="http://localhost:8080/v1", token="sk-no-key-required")

# Advanced parameters example
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a creative storyteller."},
        {"role": "user", "content": "Write a creative story"},
    ],
    temperature=0.8,
    max_tokens=200,
    top_p=0.95,
)
print(response.choices[0].message.content)

# For direct text generation
response = client.text_generation(
    "Write a creative story about space exploration",
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.95,
    repetition_penalty=1.1,
    details=True,
)
print(response.generated_text)
```

Or use the OpenAI client for generation with control over the sampling parameters:

```python
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="sk-no-key-required")

# Advanced parameters example
response = client.chat.completions.create(
    model="smollm2-1.7b-instruct",
    messages=[
        {"role": "system", "content": "You are a creative storyteller."},
        {"role": "user", "content": "Write a creative story"},
    ],
    temperature=0.8,  # Higher for more creativity
    top_p=0.95,  # Nucleus sampling probability
    frequency_penalty=0.5,  # Reduce repetition of frequent tokens
    presence_penalty=0.5,  # Reduce repetition by penalizing tokens already present
    max_tokens=200,  # Maximum generation length
)
print(response.choices[0].message.content)
```

You can also use llama.cpp's native library for even more control:

```python
# Using llama-cpp-python package for direct model access
from llama_cpp import Llama

# Load the model
llm = Llama(
    model_path="smollm2-1.7b-instruct.Q4_K_M.gguf",
    n_ctx=4096,  # Context window size
    n_threads=8,  # CPU threads
    n_gpu_layers=0,  # GPU layers (0 = CPU only)
)

# Format prompt according to the model's expected format
prompt = """<|im_start|>system
You are a creative storyteller.
<|im_end|>
<|im_start|>user
Write a creative story
<|im_end|>
<|im_start|>assistant
"""

# Generate response with precise parameter control
output = llm(
    prompt,
    max_tokens=200,
    temperature=0.8,
    top_p=0.95,
    frequency_penalty=0.5,
    presence_penalty=0.5,
    stop=["<|im_end|>"],
)

print(output["choices"][0]["text"])
```

**vLLM**

For advanced usage with vLLM, you can use the InferenceClient:

```python
from huggingface_hub import InferenceClient

client = InferenceClient(model="http://localhost:8000/v1")

# Advanced parameters example
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a creative storyteller."},
        {"role": "user", "content": "Write a creative story"},
    ],
    temperature=0.8,
    max_tokens=200,
    top_p=0.95,
)
print(response.choices[0].message.content)

# For direct text generation
response = client.text_generation(
    "Write a creative story about space exploration",
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.95,
    details=True,
)
print(response.generated_text)
```

You can also use the OpenAI client:

```python
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

# Advanced parameters example
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=[
        {"role": "system", "content": "You are a creative storyteller."},
        {"role": "user", "content": "Write a creative story"},
    ],
    temperature=0.8,
    top_p=0.95,
    max_tokens=200,
)
print(response.choices[0].message.content)
```

vLLM also provides a native Python interface with fine-grained control:

```python
from vllm import LLM, SamplingParams

# Initialize the model with advanced parameters
llm = LLM(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    gpu_memory_utilization=0.85,
    max_num_batched_tokens=8192,
    max_num_seqs=256,
    block_size=16,
)

# Configure sampling parameters
sampling_params = SamplingParams(
    temperature=0.8,  # Higher for more creativity
    top_p=0.95,  # Consider top 95% probability mass
    max_tokens=100,  # Maximum length
    presence_penalty=1.1,  # Reduce repetition
    frequency_penalty=1.1,  # Reduce repetition
    stop=["\n\n", "###"],  # Stop sequences
)

# Generate text
prompt = "Write a creative story"
outputs = llm.generate(prompt, sampling_params)
print(outputs[0].outputs[0].text)

# For chat-style interactions
chat_prompt = [
    {"role": "system", "content": "You are a creative storyteller."},
    {"role": "user", "content": "Write a creative story"},
]
formatted_prompt = llm.get_chat_template()(chat_prompt)  # Uses model's chat template
outputs = llm.generate(formatted_prompt, sampling_params)
print(outputs[0].outputs[0].text)
```

### Advanced Generation Control

#### Token Selection and Sampling

The process of generating text involves selecting the next token at each step. This selection process can be controlled through various parameters:

1. **Raw Logits**: The initial output probabilities for each token
2. **Temperature**: Controls randomness in selection (higher = more creative)
3. **Top-p (Nucleus) Sampling**: Filters to top tokens making up X% of probability mass
4. **Top-k Filtering**: Limits selection to k most likely tokens

Here's how to configure these parameters:

**TGI**

```python
client.generate(
    "Write a creative story",
    temperature=0.8,  # Higher for more creativity
    top_p=0.95,  # Consider top 95% probability mass
    top_k=50,  # Consider top 50 tokens
    max_new_tokens=100,  # Maximum length
    repetition_penalty=1.1,  # Reduce repetition
)
```

**llama.cpp**

```python
# Via OpenAI API compatibility
response = client.completions.create(
    model="smollm2-1.7b-instruct",  # Model name (can be any string for llama.cpp server)
    prompt="Write a creative story",
    temperature=0.8,  # Higher for more creativity
    top_p=0.95,  # Consider top 95% probability mass
    frequency_penalty=1.1,  # Reduce repetition
    presence_penalty=0.1,  # Reduce repetition
    max_tokens=100,  # Maximum length
)

# Via llama-cpp-python direct access
output = llm(
    "Write a creative story",
    temperature=0.8,
    top_p=0.95,
    top_k=50,
    max_tokens=100,
    repeat_penalty=1.1,
)
```

**vLLM**

```python
params = SamplingParams(
    temperature=0.8,  # Higher for more creativity
    top_p=0.95,  # Consider top 95% probability mass
    top_k=50,  # Consider top 50 tokens
    max_tokens=100,  # Maximum length
    presence_penalty=0.1,  # Reduce repetition
)
llm.generate("Write a creative story", sampling_params=params)
```

In [ ]:
# 👀 Look (not in the course): what temperature, top-k and top-p do to ONE next-token distribution.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

gen_ckpt = "HuggingFaceTB/SmolLM2-360M-Instruct"  # the model the page serves
gen_tok = AutoTokenizer.from_pretrained(gen_ckpt)
gen_model = AutoModelForCausalLM.from_pretrained(gen_ckpt)

messages = [{"role": "user", "content": "Write a creative story"}]
prompt = gen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)  # chat format the model was trained on
ids = gen_tok(prompt, return_tensors="pt")

with torch.no_grad():
    logits = gen_model(**ids).logits[0, -1]  # raw scores for the NEXT token: one per vocabulary entry
print("vocabulary size:", logits.shape[0])

for T in [0.5, 1.0, 1.5]:
    probs = torch.softmax(logits / T, dim=-1)  # temperature divides the logits before softmax
    top = torch.topk(probs, 5)
    sorted_probs = torch.sort(probs, descending=True).values
    n_top_p = int((sorted_probs.cumsum(0) < 0.95).sum()) + 1  # how many tokens top_p=0.95 keeps
    print(f"T={T}: top-5 {[(gen_tok.decode(int(i)), round(p.item(), 3)) for p, i in zip(top.values, top.indices)]} | top_p=0.95 keeps {n_top_p} tokens")
# top_k=50 always keeps exactly 50 tokens; top_p adapts: few when the model is sure, many when it is not.

In [ ]:
# 👀 Look (not in the course): the same prompt under different sampling settings.
def generate(max_new_tokens=60, **settings):
    torch.manual_seed(0)  # fixed random draws: differences come from the settings, not luck
    out = gen_model.generate(**ids, max_new_tokens=max_new_tokens, **settings)
    return gen_tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)  # only the new tokens

print("GREEDY (always the top token):\n", generate(do_sample=False), "\n")
print("T=0.8, top_p=0.95, top_k=50 (the page's settings):\n", generate(do_sample=True, temperature=0.8, top_p=0.95, top_k=50), "\n")
print("T=1.5, no top_p / top_k cut (too random):\n", generate(do_sample=True, temperature=1.5, top_p=1.0, top_k=0))

#### Controlling Repetition

Both frameworks provide ways to prevent repetitive text generation:

**TGI**

```python
client.generate(
    "Write a varied text",
    repetition_penalty=1.1,  # Penalize repeated tokens
    no_repeat_ngram_size=3,  # Prevent 3-gram repetition
)
```

**llama.cpp**

```python
# Via OpenAI API
response = client.completions.create(
    model="smollm2-1.7b-instruct",
    prompt="Write a varied text",
    frequency_penalty=1.1,  # Penalize frequent tokens
    presence_penalty=0.8,  # Penalize tokens already present
)

# Via direct library
output = llm(
    "Write a varied text",
    repeat_penalty=1.1,  # Penalize repeated tokens
    frequency_penalty=0.5,  # Additional frequency penalty
    presence_penalty=0.5,  # Additional presence penalty
)
```

**vLLM**

```python
params = SamplingParams(
    presence_penalty=0.1,  # Penalize token presence
    frequency_penalty=0.1,  # Penalize token frequency
)
```

In [ ]:
# 👀 Look (not in the course): repetition controls. Greedy decoding on a small model often loops.
rep_ids = gen_tok("The best thing about the city is", return_tensors="pt")

def complete(**settings):
    out = gen_model.generate(**rep_ids, max_new_tokens=60, do_sample=False, **settings)
    return gen_tok.decode(out[0, rep_ids["input_ids"].shape[1]:], skip_special_tokens=True)

print("no penalty:\n", complete(), "\n")
print("repetition_penalty=1.3 (shrink the logit of every token already in the text):\n", complete(repetition_penalty=1.3), "\n")
print("no_repeat_ngram_size=3 (ban any 3-token sequence from appearing twice):\n", complete(no_repeat_ngram_size=3))

#### Length Control and Stop Sequences

You can control generation length and specify when to stop:

**TGI**

```python
client.generate(
    "Generate a short paragraph",
    max_new_tokens=100,
    min_new_tokens=10,
    stop_sequences=["\n\n", "###"],
)
```

**llama.cpp**

```python
# Via OpenAI API
response = client.completions.create(
    model="smollm2-1.7b-instruct",
    prompt="Generate a short paragraph",
    max_tokens=100,
    stop=["\n\n", "###"],
)

# Via direct library
output = llm("Generate a short paragraph", max_tokens=100, stop=["\n\n", "###"])
```

**vLLM**

```python
params = SamplingParams(
    max_tokens=100,
    min_tokens=10,
    stop=["###", "\n\n"],
    ignore_eos=False,
    skip_special_tokens=True,
)
```

In [ ]:
# 👀 Look (not in the course): length and stop controls.
print("max_new_tokens=15 (hard cut, mid-sentence):\n", generate(max_new_tokens=15, do_sample=False), "\n")
print('stop_strings=["."] (stop at the first full stop):\n', generate(do_sample=False, stop_strings=["."], tokenizer=gen_tok))

### Memory Management

Both frameworks implement advanced memory management techniques for efficient inference.

**TGI**

TGI uses Flash Attention 2 and continuous batching:

```sh
# Docker deployment with memory optimization
docker run --gpus all -p 8080:80 \
    --shm-size 1g \
    ghcr.io/huggingface/text-generation-inference:latest \
    --model-id HuggingFaceTB/SmolLM2-1.7B-Instruct \
    --max-batch-total-tokens 8192 \
    --max-input-length 4096
```

**llama.cpp**

llama.cpp uses quantization and optimized memory layout:

```sh
# Server with memory optimizations
./server \
    -m smollm2-1.7b-instruct.Q4_K_M.gguf \
    --host 0.0.0.0 \
    --port 8080 \
    -c 2048 \               # Context size
    --threads 4 \           # CPU threads
    --n-gpu-layers 32 \     # Use more GPU layers for larger models
    --mlock \               # Lock memory to prevent swapping
    --cont-batching         # Enable continuous batching
```

For models too large for your GPU, you can use CPU offloading:

```sh
./server \
    -m smollm2-1.7b-instruct.Q4_K_M.gguf \
    --n-gpu-layers 20 \     # Keep first 20 layers on GPU
    --threads 8             # Use more CPU threads for CPU layers
```

**vLLM**

vLLM uses PagedAttention for optimal memory management:

```python
from vllm.engine.arg_utils import AsyncEngineArgs

engine_args = AsyncEngineArgs(
    model="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    gpu_memory_utilization=0.85,
    max_num_batched_tokens=8192,
    block_size=16,
)

llm = LLM(engine_args=engine_args)
```

In [ ]:
# 👀 Look (not in the course): the KV cache, the thing PagedAttention manages.
# With the cache, each new token reuses the stored keys/values of all earlier tokens.
# Without it, every step recomputes them for the whole sequence so far.
import time

outs = {}
for use_cache in [True, False]:
    start = time.time()
    outs[use_cache] = gen_model.generate(**ids, max_new_tokens=100, do_sample=False, use_cache=use_cache)
    print(f"use_cache={use_cache}: {time.time() - start:.1f} s")
print("same text both ways:", torch.equal(outs[True], outs[False]))  # the cache is a pure speed-up, not an approximation

cfg = gen_model.config
head_dim = cfg.hidden_size // cfg.num_attention_heads
per_token = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * 2  # K and V, every layer, fp16 = 2 bytes
print(f"KV cache: {per_token / 1024:.0f} KB per token -> {per_token * 4096 / 1e6:.0f} MB for ONE 4096-token request")

In [ ]:
# 👀 Look (not in the course): why llama.cpp quantizes. Memory = parameters x bytes per parameter.
n = gen_model.num_parameters()
print(f"{n / 1e6:.0f}M parameters")
for name, bytes_per in [("fp32", 4), ("fp16", 2), ("int8", 1), ("4-bit (Q4)", 0.5)]:
    print(f"{name:12} {n * bytes_per / 1e9:.2f} GB")
# Same arithmetic for SmolLM2-1.7B (the page's GGUF): ~6.8 GB in fp32 vs ~1 GB in Q4.

### Resources

- [Text Generation Inference Documentation](https://huggingface.co/docs/text-generation-inference)
- [TGI GitHub Repository](https://github.com/huggingface/text-generation-inference)
- [vLLM Documentation](https://vllm.readthedocs.io/)
- [vLLM GitHub Repository](https://github.com/vllm-project/vllm)
- [PagedAttention Paper](https://arxiv.org/abs/2309.06180)
- [llama.cpp GitHub Repository](https://github.com/ggerganov/llama.cpp)
- [llama-cpp-python Repository](https://github.com/abetlen/llama-cpp-python)